# 06 — Human Evaluation Template

**RLHF Pipeline for Compact Open-Source LLMs**

This notebook provides a template for blind pairwise human evaluation of model outputs. Human evaluation is the gold standard for assessing RLHF alignment quality.

## Workflow

1. **Export** — Generate a CSV with pairs of model outputs (randomised order)
2. **Annotate** — Evaluator reads each pair and records a preference (A, B, or tie)
3. **Import** — Load annotations back into this notebook
4. **Aggregate** — Compute win rates and summary statistics

## 6.1 Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import random
import pandas as pd
import numpy as np

from src.data_utils import set_seed
from src.metrics import human_eval_agreement
from src.inference import load_samples

set_seed(42)

SAMPLES_DIR = PROJECT_ROOT / "outputs" / "samples"
TABLES_DIR  = PROJECT_ROOT / "outputs" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 6.2 Export Blind Pairwise Evaluation CSV

This cell creates a CSV where each row has a prompt and two responses (A and B) in randomised order. The evaluator does not know which model produced which response.

In [ ]:
comparison_path = SAMPLES_DIR / "model_comparison.csv"

if comparison_path.exists():
    df = load_samples(comparison_path)
    
    # Pivot to get SFT and PPO responses per prompt
    model_a_name = "sft"
    model_b_name = "ppo"
    
    available = df["model_name"].unique()
    if model_a_name not in available or model_b_name not in available:
        print(f"Available models: {available}")
        print(f"Need both '{model_a_name}' and '{model_b_name}' — adjust names above.")
    else:
        a_df = df[df["model_name"] == model_a_name][["prompt", "response"]].rename(columns={"response": "response_real_a"})
        b_df = df[df["model_name"] == model_b_name][["prompt", "response"]].rename(columns={"response": "response_real_b"})
        
        merged = a_df.merge(b_df, on="prompt")
        
        # Randomise which appears as A vs B
        random.seed(42)
        eval_rows = []
        for _, row in merged.iterrows():
            swap = random.random() < 0.5
            eval_rows.append({
                "prompt": row["prompt"],
                "response_A": row["response_real_b"] if swap else row["response_real_a"],
                "response_B": row["response_real_a"] if swap else row["response_real_b"],
                "_actual_A": model_b_name if swap else model_a_name,
                "_actual_B": model_a_name if swap else model_b_name,
                "preference": "",  # To be filled by evaluator: A, B, or tie
            })
        
        eval_df = pd.DataFrame(eval_rows)
        
        # Export version WITHOUT ground truth (for evaluator)
        blind_df = eval_df[["prompt", "response_A", "response_B", "preference"]]
        blind_path = SAMPLES_DIR / "human_eval_blind.csv"
        blind_df.to_csv(blind_path, index=False)
        print(f"Blind evaluation CSV exported to {blind_path} ({len(blind_df)} pairs)")
        
        # Export version WITH ground truth (for analysis)
        key_path = SAMPLES_DIR / "human_eval_key.csv"
        eval_df.to_csv(key_path, index=False)
        print(f"Answer key CSV exported to {key_path}")
else:
    print("No model comparison data found. Run notebook 05 section 5.3 first.")

## 6.3 Import Annotated Results

After the evaluator fills in the `preference` column in `human_eval_blind.csv`, load it back here.

In [ ]:
annotated_path = SAMPLES_DIR / "human_eval_blind.csv"  # or a renamed copy

if annotated_path.exists():
    annotated = pd.read_csv(annotated_path)
    filled = annotated[annotated["preference"].notna() & (annotated["preference"] != "")]
    print(f"Loaded {len(filled)} annotated rows out of {len(annotated)} total")
    
    if len(filled) > 0:
        display(filled[["preference"]].value_counts())
    else:
        print("No annotations found yet. Fill in the 'preference' column with A, B, or tie.")
else:
    print("No annotated file found. Export it first in section 6.2.")

## 6.4 Aggregate Results

In [ ]:
if annotated_path.exists():
    annotated = pd.read_csv(annotated_path)
    filled = annotated[annotated["preference"].notna() & (annotated["preference"] != "")].copy()
    
    if len(filled) > 0:
        # Load the answer key to map A/B back to model names
        key_path = SAMPLES_DIR / "human_eval_key.csv"
        if key_path.exists():
            key_df = pd.read_csv(key_path)
            # Merge to get actual model names
            merged = filled.merge(
                key_df[["prompt", "_actual_A", "_actual_B"]],
                on="prompt", how="left",
            )
            
            # Map preferences to actual model names
            def resolve_preference(row):
                p = str(row["preference"]).strip().upper()
                if p == "A":
                    return row["_actual_A"]
                elif p == "B":
                    return row["_actual_B"]
                return "tie"
            
            merged["winner"] = merged.apply(resolve_preference, axis=1)
            
            # Summary
            counts = merged["winner"].value_counts()
            total = len(merged)
            print("\n=== Human Evaluation Win Rates ===")
            for model, count in counts.items():
                print(f"  {model}: {count}/{total} ({count/total*100:.1f}%)")
            
            # Save summary
            summary = {k: v / total for k, v in counts.items()}
            summary["total"] = total
            pd.DataFrame([summary]).to_csv(TABLES_DIR / "human_eval_summary.csv", index=False)
        else:
            print("Answer key not found — cannot map A/B to model names.")
    else:
        print("No annotations to aggregate.")
else:
    print("TO BE FILLED AFTER HUMAN EVALUATION")

## 6.5 Discussion

_Fill in once annotations are aggregated:_

- How does human preference align with the reward model's scores?
- Are there prompts where annotators systematically disagree?
- Do annotators prefer longer or more concise responses?
- Are there cases where the reward model prefers the opposite of the human evaluator?

## Summary

| Output | Location |
|---|---|
| Blind evaluation CSV (for annotator) | `outputs/samples/human_eval_blind.csv` |
| Answer key CSV | `outputs/samples/human_eval_key.csv` |
| Human evaluation summary | `outputs/tables/human_eval_summary.csv` |